<a href="https://colab.research.google.com/github/iamimpeccable/RetrievalAugmentedGeneration-Projects/blob/main/RAGProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-community chromadb sentence-transformers pypdf openai langchain-openai

In [ ]:
from google.colab import files
uploaded = files.upload() # ← a file picker will appear
pdf_filename = list(uploaded.keys())[0]
print(f"✅ Uploaded: {pdf_filename}")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the PDF
loader = PyPDFLoader(pdf_filename)
pages = loader.load()

# Split into chunks (500 chars, 50 chars overlap so context isn't lost)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)

print(f"✅ PDF has {len(pages)} pages → split into {len(chunks)} chunks")
print(f"\nExample chunk:\n{chunks[0].page_content[:300]}")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
# Load a free embedding model (downloads ~90MB, runs on CPU)
print("Loading embedding model... (takes ~30 seconds first time)")
embeddings = HuggingFaceEmbeddings( model_name="sentence-transformers/all-MiniLM-L6-v2" )
# Embed all chunks and store in ChromaDB
vectorstore = Chroma.from_documents(chunks, embeddings)
print(f"✅ Embedded {len(chunks)} chunks into ChromaDB")

In [ ]:
!pip install -q transformers
!pip install -q langchain-huggingface

In [ ]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

# Initialize a Hugging Face pipeline with a small, free model
# This model runs locally and does not require an API key.
# It might be slow on CPU for larger models.
print("Loading Hugging Face model...")
hf_pipeline = pipeline(
    "text-generation",
    model="distilgpt2", # A small, fast model for demonstration
    tokenizer="distilgpt2",
    max_new_tokens=100
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)
print("✅ Hugging Face model loaded.")

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(
        search_kwargs={"k": 3}  # top 3 chunks
    ),
    return_source_documents=True
)

# Ask a question!
question = "What is the anomaly?"  # ← change this

result = qa_chain.invoke({
    "query": question
})

print(f"Question: {question}")
print(f"\nAnswer: {result['result']}")
print(f"\n--- Retrieved from {len(result['source_documents'])} chunks ---")